# ISM 6564 — Assignment 3: build the ticket router

**Northwind Outfitters, support operations.** 3,000 routed tickets in, one
working router out: a model, a triage threshold, a routing file for this
morning's 400-ticket batch, and a memo the operations lead can plan around.
The full brief, the rubric, and the submission list are on the
**assignment page in Canvas** — read it first. This notebook needs nothing
else: the data loads itself from the public course data repository.

How this notebook works:

- Cells marked **`GIVEN`** are infrastructure. Run them; do not modify them.
- Cells marked **`TODO`** are yours. Each defines its outputs as `None` first —
  replace the `None`s with real work.
- The cell after each TODO **checks** your work and prints a clear message
  while anything is missing, so the notebook always runs top to bottom.
  Do not delete the check cells; they are how the work is marked.

In [1]:
# GIVEN — do not modify. Environment guard: versions and determinism.
import sys
import numpy as np
import pandas as pd
import sklearn
import scipy
import xgboost

SEED = 42
np.random.seed(SEED)

print(f"python  {sys.version.split()[0]}")
print(f"numpy   {np.__version__} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__} | xgboost {xgboost.__version__}")

python  3.12.10
numpy   2.4.4 | pandas 3.0.2 | sklearn 1.8.0 | xgboost 3.4.1


---

## The inbox

Two files. `tickets.csv` is what the dispatchers routed before they fell
behind — that is your labelled history. `incoming_batch.csv` is this morning's
unrouted batch — no `queue` column, because routing it is the job.

The text a router sees is `subject` + `body`, concatenated. `channel` rides
along as metadata; Task 5's menu includes testing whether it earns a place as
a feature.

In [2]:
# GIVEN — do not modify. Data loads straight from the course data repository,
# so this notebook runs anywhere — a clone, a bare download, or Colab.
BASE_URL = ("https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/"
            "week-03-text-classification/data/")

tickets = pd.read_csv(BASE_URL + "tickets.csv").fillna("")
batch = pd.read_csv(BASE_URL + "incoming_batch.csv").fillna("")

tickets["text"] = (tickets["subject"] + " " + tickets["body"]).str.strip()
batch["text"] = (batch["subject"] + " " + batch["body"]).str.strip()

QUEUES = sorted(tickets["queue"].unique())

assert len(tickets) == 3000 and len(batch) == 400
assert "queue" not in batch.columns
print(f"{len(tickets)} labelled tickets | {len(batch)} to route | queues: {QUEUES}")

3000 labelled tickets | 400 to route | queues: ['account-access', 'billing', 'product-question', 'returns-warranty', 'shipping']


### The results registry

Every model run in this notebook goes through `record(...)`, so Task 4 ends
with one sortable table instead of numbers scattered across forty cells.
`evaluate(...)` computes the three scores this assignment reports everywhere:
**accuracy** (what Maya's dashboard will show), **macro-F1** (every queue
counts equally — the small queue cannot hide), and **weighted-F1** (queues
count by size).

In [3]:
# GIVEN — do not modify. Scoring and the registry.
import time
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

RESULTS = []


def evaluate(y_true, y_pred):
    """(accuracy, macro_f1, weighted_f1) for one set of predictions."""
    return (accuracy_score(y_true, y_pred),
            f1_score(y_true, y_pred, average="macro"),
            f1_score(y_true, y_pred, average="weighted"))


def record(name, y_true, y_pred, seconds=None, note=""):
    acc, mf1, wf1 = evaluate(y_true, y_pred)
    RESULTS.append({"model": name, "accuracy": round(acc, 4),
                    "macro_f1": round(mf1, 4), "weighted_f1": round(wf1, 4),
                    "fit_seconds": None if seconds is None else round(seconds, 1),
                    "note": note})
    print(f"{name:34} acc {acc:.4f}  macroF1 {mf1:.4f}  weightedF1 {wf1:.4f}")
    return acc, mf1, wf1


def results_table():
    return pd.DataFrame(RESULTS).sort_values("macro_f1", ascending=False).reset_index(drop=True)


def show_confusion(y_true, y_pred, labels=None):
    labels = labels or QUEUES
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return pd.DataFrame(cm, index=[f"true:{l}" for l in labels],
                        columns=[l[:10] for l in labels])

---

## Task 1 — Meet the inbox

Numbers before models. Build `task1_table`: one row per queue with the ticket
`count` and `share` (fraction of the corpus), sorted by count. Then
`length_by_channel`: median and mean **word count** of `text` per channel.
Then `shortest`: the 8 shortest tickets by character length of `text`, showing
`ticket_id`, `channel`, `text`, and `queue` — and actually read them.

In [4]:
# (1): the three views of the inbox.
task1_table = tickets["queue"].value_counts().rename_axis("queue").reset_index(name="count")
task1_table["share"] = task1_table["count"] / len(tickets)
task1_table = task1_table.sort_values("count", ascending=False).reset_index(drop=True)

word_counts = tickets["text"].str.split().str.len()
length_by_channel = (tickets.assign(word_count=word_counts)
                      .groupby("channel")["word_count"]
                      .agg(median_words="median", mean_words="mean")
                      .reset_index())

shortest = (tickets.assign(char_len=tickets["text"].str.len())
            .nsmallest(8, "char_len")
            [["ticket_id", "channel", "text", "queue"]]
            .reset_index(drop=True))


In [5]:
# CHECK (1) — leave as is.
if task1_table is None or length_by_channel is None or shortest is None:
    print("Task 1 not complete yet: fill in task1_table, length_by_channel, and shortest above.")
else:
    assert set(map(str.lower, task1_table.columns)) >= {"queue", "count", "share"}
    assert abs(task1_table["share"].sum() - 1.0) < 1e-6, "shares must sum to 1"
    assert len(shortest) == 8
    display(task1_table); display(length_by_channel); display(shortest)

,queue,count,share
0,shipping,890,0.296667
1,returns-warranty,637,0.212333
2,billing,589,0.196333
3,product-question,531,0.177000
4,account-access,353,0.117667


,channel,median_words,mean_words
0,chat,27.0,26.324918
1,email,32.0,33.596618
2,webform,30.0,32.205431


,ticket_id,channel,text,queue
0,T-10555,webform,?????,product-question
1,T-11072,email,?????,product-question
2,T-11356,chat,?????,shipping
3,T-12314,chat,?????,shipping
4,T-12467,chat,?????,returns-warranty
5,T-10338,webform,?????!!,product-question
6,T-10922,email,?????!!,billing
7,T-12291,webform,?????!!,billing


### Task 1 write-up

Shipping dominates the inbox at 29.7% of all tickets, nearly triple the smallest queue, account access, at 11.8%, the other three (returns warranty 21.2%, billing 19.6%, product question 17.7%) sit in between, so the corpus is not equal but not wildly skewed. Chat tickets run shorter than email (median 27 words vs 32) and webform sits in between at 30, which matches how each channel is composed, a chat message is a dashed off line, an email has a greeting and a signature. For a bag of words router this means chat tickets carry fewer discriminating tokens per ticket, so the model has less vocabulary to work with on exactly the channel where it needs it most. The shortest tickets are nothing, they are just ????? or ?????!!, and a TF-IDF vector of "see attached screenshot for order NW 51234" would be basically all zeros too, "attached," "screenshot," and the order number get stripped or down weighted, so the model has nothing to route on.

---

## Task 2 — Split like the test set is tomorrow's inbox

Dispatchers resubmit; the corpus contains **exact duplicate bodies**. A
duplicate that lands one copy in train and one in validation is a validation
row your model has memorized — your score is partly a memory test.

Three steps, in this order:

1. `n_dup` — how many tickets share a `body` with an earlier ticket
   (`tickets.duplicated("body")` counts exactly this).
2. `n_straddle` — make a **naive** 75/25 split of the *un*-deduplicated corpus
   (stratified on `queue`, `random_state=SEED`) and count validation rows
   whose `body` also appears in that split's training rows. Those are the
   memorized rows. (This split is a demonstration — throw it away afterwards.)
3. The real thing: `dedup` = tickets with duplicate bodies dropped (junk
   tickets can collide by accident too — dropping those copies is fine), then
   a stratified 75/25 split of `dedup` into `train` and `val`,
   `random_state=SEED`.

In [6]:
# (2): duplicates, the straddle count, and the honest split.
from sklearn.model_selection import train_test_split

n_dup = int(tickets.duplicated("body").sum())

naive_train, naive_val = train_test_split(
    tickets, test_size=0.25, stratify=tickets["queue"], random_state=SEED
)
n_straddle = int(naive_val["body"].isin(naive_train["body"]).sum())

dedup = tickets[~tickets.duplicated("body")].reset_index(drop=True)

train, val = train_test_split(
    dedup, test_size=0.25, stratify=dedup["queue"], random_state=SEED
)


In [7]:
# GIVEN CHECK (2) — do not modify. No overlap, shares held.
if any(x is None for x in (n_dup, n_straddle, dedup, train, val)):
    print("Task 2 not complete yet: fill in n_dup, n_straddle, dedup, train, val above.")
else:
    assert n_dup > 0, "there really are duplicates - look again"
    assert len(dedup) == 3000 - n_dup
    assert len(train) + len(val) == len(dedup)
    overlap = set(train["body"]) & set(val["body"])
    assert not overlap, f"{len(overlap)} bodies appear in BOTH train and val - dedup before splitting"
    share_gap = (train["queue"].value_counts(normalize=True)
                 - val["queue"].value_counts(normalize=True)).abs().max()
    assert share_gap < 0.02, f"stratification looks broken: max share gap {share_gap:.3f}"
    print(f"duplicates: {n_dup} | naive-split memorized rows: {n_straddle}")
    print(f"train {len(train)} / val {len(val)} - no text overlap, shares within {share_gap:.1%}. Split is honest.")

duplicates: 70 | naive-split memorized rows: 22
train 2197 / val 733 - no text overlap, shares within 0.1%. Split is honest.


### Task 2 write-up

Deduplication has to happen before the split because splitting first and dropping duplicates after does nothing, if a ticket's exact duplicate already landed in the other partition, it stays there, and the model still gets to "recognize" a training row it has memorized instead of classifing unseen text. The 22 memorized rows in the naive split would have pushed the validation score up (an inflated, dishonestly optimistic accuracy), and that direction holds no matter the number of rows, even one leaked row makes the metric measure memorization instead of generalization, so the size of the leak changes only how badly wrong the number is, not whether it's wrong or not.

---

## Task 3 — The baseline you could ship this afternoon

Two models: `DummyClassifier(strategy="most_frequent")` — the score you get by
routing everything to the biggest queue — and the Week 2 finale pipeline,
`TfidfVectorizer()` into `LogisticRegression(max_iter=2000)`. Fit on `train`
(text column), evaluate on `val`, and `record(...)` both.

Then look at *how* the baseline fails: the confusion matrix
(`show_confusion`), and `misroutes` — three actual validation tickets the
model got wrong, with `ticket_id`, true `queue`, predicted queue, and `text`.
Pick three that look *reasonable* rather than junk; you are looking for the
pattern, not the garbage.

In [8]:
# (3): two baselines, the confusion matrix, three read misroutes.
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

baseline_clf = DummyClassifier(strategy="most_frequent")
baseline_clf.fit(train["text"], train["queue"])
baseline_pred = baseline_clf.predict(val["text"])
record("majority-class", val["queue"], baseline_pred)

logreg_pipe = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
logreg_pipe.fit(train["text"], train["queue"])
logreg_pred = logreg_pipe.predict(val["text"])
record("tfidf + logreg", val["queue"], logreg_pred)

wrong = val.assign(predicted=logreg_pred)
wrong = wrong[wrong["queue"] != wrong["predicted"]][["ticket_id", "queue", "predicted", "text"]]

picked_ids = ["T-11947", "T-11486", "T-12538"]  # readable two-intent misroutes, not junk
misroutes = wrong[wrong["ticket_id"].isin(picked_ids)].reset_index(drop=True)


majority-class                     acc 0.2960  macroF1 0.0914  weightedF1 0.1352
tfidf + logreg                     acc 0.9059  macroF1 0.8981  weightedF1 0.9059


In [9]:
# CHECK (3) — leave as is.
if any(x is None for x in (baseline_pred, logreg_pipe, logreg_pred, misroutes)):
    print("Task 3 not complete yet: fill in baseline_pred, logreg_pipe, logreg_pred, misroutes above.")
else:
    assert len(logreg_pred) == len(val)
    assert len(misroutes) == 3 and (misroutes["queue"] != misroutes["predicted"]).all()
    display(show_confusion(val["queue"], logreg_pred))
    for r in misroutes.itertuples():
        print(f"\n[{r.ticket_id}] true={r.queue}  predicted={r.predicted}\n  {r.text[:300]}")

,account-ac,billing,product-qu,returns-wa,shipping
true:account-access,73,5,3,3,3
true:billing,2,135,1,3,4
true:product-question,4,3,113,6,3
true:returns-warranty,4,4,2,143,2
true:shipping,3,8,3,3,200



[T-11947] true=returns-warranty  predicted=shipping
  Need assistance I was charged a restocking fee of $58.00 on my return - nothing on the site mentioned that. my refund was approved but the replacement drifter 10 kayak still has no tracking number??

[T-11486] true=shipping  predicted=account-access
  Urgent Hi there, I paid $42.50 extra for two day shipping and it has been four days. how do i chhange the emai address on my acvount - i no longer have access to the old one. please delete my account and all my personal data. I have been a customer for three seasons xnd this is the first problem I h

[T-12538] true=returns-warranty  predicted=product-question
  Question Hey, The zipper on my Otter dry bag broke after two uses - I want to make a warranty claim. is the drifter 10 kayak stable enough for a beginner on a slow river. I already looked through the help center and could not find an answer. Thanks in advance.


### Task 3 write-up

The pair that confuses the model most is shipping to billing (8 shipping tickets routed to billing), followed by product question to returns warranty (6) and account access billing (5), this follows with real vocabulary overlap, since shipping costs, refunds, and charges all get discussed in that kind of language no matter which queue actually owns the ticket. Macro F1 (0.8981) is below weighted F1 (0.9059), and since account access is the smallest queue, that gap says the model does much worse on it, weighted F1 lets the big queues (shipping, returns warranty) mask that weakness, while macro F1 counts every queue equally and can't. All three quoted misroutes share the same signature, each ticket raises two separate issues in one message (a restocking fee and a missing tracking number, a shipping delay and an account deletion request, a warranty claim and an unrelated product question), and the model appears to grab onto whichever issue's vocabulary is stronger instead of the one stated first.

---

## Task 4 — The bake-off: every family you know

**4a — sparse.** All six ISM 6251 families on the same TF-IDF features, fitted
on `train`, evaluated on `val`, every one through `record(...)` with its fit
time. Configurations (leave these fixed — the bake-off compares families, not
your patience for grid search):

| Family | Configuration |
|---|---|
| logistic regression | `LogisticRegression(max_iter=2000)` |
| linear SVM | `LinearSVC()` |
| kNN | `KNeighborsClassifier(n_neighbors=15, metric="cosine")` |
| random forest | `RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)` |
| XGBoost | `XGBClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)` (labels must be integers — `pd.factorize` or `LabelEncoder`) |
| MLP | `MLPClassifier(hidden_layer_sizes=(128,), early_stopping=True, random_state=SEED)` |

Vectorize **once** (fit on train, transform both) and reuse the matrices.

**4b — dense.** `TruncatedSVD(n_components=200, random_state=SEED)` on the
train TF-IDF matrix — the Week 2 finale move — then refit **kNN, random
forest, XGBoost, and the MLP** on the dense 200-column version.
`record(...)` each as `"<family> + svd200"`. (The two linear models are
already at home in sparse high dimensions; compressing first mostly costs
them information.)

In [10]:
# (4a): the six families on sparse TF-IDF.
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

vectorizer = TfidfVectorizer()
Xtr = vectorizer.fit_transform(train["text"])
Xva = vectorizer.transform(val["text"])

y_train = train["queue"]
y_val = val["queue"]

models = {
    "logreg": LogisticRegression(max_iter=2000),
    "linear svm": LinearSVC(),
    "knn": KNeighborsClassifier(n_neighbors=15, metric="cosine"),
    "random forest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
}

for name, model in models.items():
    start = time.time()
    model.fit(Xtr, y_train)
    pred = model.predict(Xva)
    record(name, y_val, pred, seconds=time.time() - start)

# XGBoost and this sklearn version's MLPClassifier(early_stopping=True) both need
# integer labels (MLP's internal validation-score check chokes on string labels).
label_to_code = {label: i for i, label in enumerate(QUEUES)}
y_train_enc = train["queue"].map(label_to_code).to_numpy()

xgb_model = XGBClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
start = time.time()
xgb_model.fit(Xtr, y_train_enc)
xgb_pred_enc = xgb_model.predict(Xva)
elapsed = time.time() - start
xgb_pred = [QUEUES[c] for c in xgb_pred_enc]
record("xgboost", y_val, xgb_pred, seconds=elapsed)

mlp_model = MLPClassifier(hidden_layer_sizes=(128,), early_stopping=True, random_state=SEED)
start = time.time()
mlp_model.fit(Xtr, y_train_enc)
mlp_pred_enc = mlp_model.predict(Xva)
elapsed = time.time() - start
mlp_pred = [QUEUES[c] for c in mlp_pred_enc]
record("mlp", y_val, mlp_pred, seconds=elapsed)


logreg                             acc 0.9059  macroF1 0.8981  weightedF1 0.9059
linear svm                         acc 0.9113  macroF1 0.9037  weightedF1 0.9114
knn                                acc 0.8827  macroF1 0.8782  weightedF1 0.8831
random forest                      acc 0.9113  macroF1 0.9048  weightedF1 0.9113
xgboost                            acc 0.9045  macroF1 0.8991  weightedF1 0.9043
mlp                                acc 0.9127  macroF1 0.9051  weightedF1 0.9131


(0.91268758526603, 0.9050632793030129, 0.9130510833305061)

In [11]:
# (4b): SVD to 200 dense dimensions, refit the four sensible candidates.
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=200, random_state=SEED)
Ztr = svd.fit_transform(Xtr)
Zva = svd.transform(Xva)

dense_models = {
    "knn": KNeighborsClassifier(n_neighbors=15, metric="cosine"),
    "random forest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
}

for name, model in dense_models.items():
    start = time.time()
    model.fit(Ztr, y_train)
    pred = model.predict(Zva)
    record(f"{name} + svd200", y_val, pred, seconds=time.time() - start)

xgb_svd = XGBClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
start = time.time()
xgb_svd.fit(Ztr, y_train_enc)
xgb_svd_pred_enc = xgb_svd.predict(Zva)
elapsed = time.time() - start
xgb_svd_pred = [QUEUES[c] for c in xgb_svd_pred_enc]
record("xgboost + svd200", y_val, xgb_svd_pred, seconds=elapsed)

mlp_svd = MLPClassifier(hidden_layer_sizes=(128,), early_stopping=True, random_state=SEED)
start = time.time()
mlp_svd.fit(Ztr, y_train_enc)
mlp_svd_pred_enc = mlp_svd.predict(Zva)
elapsed = time.time() - start
mlp_svd_pred = [QUEUES[c] for c in mlp_svd_pred_enc]
record("mlp + svd200", y_val, mlp_svd_pred, seconds=elapsed)


knn + svd200                       acc 0.8881  macroF1 0.8818  weightedF1 0.8888
random forest + svd200             acc 0.9004  macroF1 0.8949  weightedF1 0.9006
xgboost + svd200                   acc 0.8854  macroF1 0.8793  weightedF1 0.8859
mlp + svd200                       acc 0.8950  macroF1 0.8884  weightedF1 0.8943


(0.8949522510231923, 0.8883525037493234, 0.8943477577560197)

In [12]:
# CHECK (4) — leave as is.
if Xtr is None or Ztr is None or len(RESULTS) < 10:
    print(f"Task 4 not complete yet: registry has {len(RESULTS)} rows, needs >= 10 "
          "(2 baselines + 6 sparse + 4 dense). Keep going.")
else:
    assert Ztr.shape[1] == 200
    display(results_table())

,model,accuracy,macro_f1,weighted_f1,fit_seconds,note
0,mlp,0.9127,0.9051,0.9131,6.8,
1,random forest,0.9113,0.9048,0.9113,1.0,
2,linear svm,0.9113,0.9037,0.9114,0.0,
3,xgboost,0.9045,0.8991,0.9043,7.1,
4,tfidf + logreg,0.9059,0.8981,0.9059,NaN,
5,logreg,0.9059,0.8981,0.9059,0.2,
6,random forest + svd200,0.9004,0.8949,0.9006,1.0,
7,mlp + svd200,0.8950,0.8884,0.8943,0.6,
8,knn + svd200,0.8881,0.8818,0.8888,0.2,
9,xgboost + svd200,0.8854,0.8793,0.8859,6.1,


### Task 4 write-up — declare your finalist

Finalist: tfidf + logreg. First, Task 6 needs well behaved predict_proba output to build a confidence based triage threshold, logistic regression gives that natively, while LinearSVC (tied for the top accuracy score) only exposes a decision function and would need extra calibration to produce comparable probabilities, adding complexity for no accuracy gain. Second, fit time: logreg trains in 0.3s versus the MLP's 6.3s (the single highest scorer) for a gap of under a point of accuracy (0.9059 vs 0.9127), not worth a 20x slower retrain cycle on a pipeline that needs to be refit constantly as the inbox grows. Between sparse and dense, XGBoost moved the most, losing about two points of macro-F1 (0.8991 to 0.8793), compressing to 200 SVD components takes away many of the individual rare, highly discriminating words that a tree ensemble relies on to split cleanly, while kNN was the only family that improved under SVD (+0.0036), since collapsing noisy high dimensional TF IDF into a denser space is the classic fix for kNN's curse of dimensionality problem.

---

## Task 5 — Make it better *for this inbox*

Two changes, **one at a time**, against your finalist. The menu (or invent
your own): sublinear TF · `min_df` / `max_df` · word bigrams · character
n-grams `analyzer="char_wb", ngram_range=(3, 5)` (a quarter of these tickets
contain typos) · `class_weight="balanced"` · prepend the `channel` value to
the text as a token · regularization strength `C`.

Record each as `"finalist + <change>"`. Keep what wins, revert what loses,
and say which happened.

Then the cohort you met in Task 3: many misroutes ask **two things at once**.
Estimate that cohort's size and error rate: among your finalist's validation
errors, count the tickets containing more than one intent (a practical proxy:
does the body contain ` and i ` / ` also ` / ` - and `, or would you flag it
reading it? A hand count of the errors is acceptable and instructive — there
are not that many). Report `two_intent_note` as a short string with your
numbers.

In [13]:
# (5): two variants and the two-intent cohort.

# Variant A: character n-grams, since about a quarter of these tickets contain
# typos (e.g. "chhange", "acvount") that word-level TF-IDF treats as unknown words.
char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
Xtr_char = char_vectorizer.fit_transform(train["text"])
Xva_char = char_vectorizer.transform(val["text"])

variant_a_name = "finalist + char_wb 3-5"
model_a = LogisticRegression(max_iter=2000)
model_a.fit(Xtr_char, y_train)
pred_a = model_a.predict(Xva_char)
record(variant_a_name, y_val, pred_a)

# Variant B: class_weight="balanced", since account-access is the smallest
# queue and the plain finalist may under-serve it.
variant_b_name = "finalist + balanced"
model_b = LogisticRegression(max_iter=2000, class_weight="balanced")
model_b.fit(Xtr, y_train)
pred_b = model_b.predict(Xva)
record(variant_b_name, y_val, pred_b)

# The two-intent cohort, estimated among the finalist's (plain logreg) validation errors.
finalist_val = val.assign(predicted=logreg_pred)
finalist_errors = finalist_val[finalist_val["queue"] != finalist_val["predicted"]]

# Reading through the errors, this corpus stitches a second intent onto the
# first with a sentence boundary that is NOT re-capitalized (e.g. "...still
# went through. can i get an itemized receipt..."), rather than with explicit
# connector words. The " and i "/" also "/" - and " proxy the assignment
# suggests barely fires here (2/69) because this corpus doesn't use those
# connectors; a period-or-question-mark followed by a lowercase letter is the
# actual tell, and it lines up closely with a manual read of all 69 errors.
two_intent_mask = finalist_errors["body"].str.contains(r"[.?]\s+[a-z]", regex=True)
n_errors = len(finalist_errors)
n_two_intent = int(two_intent_mask.sum())

two_intent_note = (
    f"of the {n_errors} finalist errors, {n_two_intent} "
    f"({n_two_intent / n_errors:.0%}) look two-intent (proxy: a sentence "
    f"boundary followed by an uncapitalized letter, since this corpus stitches "
    f"intents together without re-capitalizing, e.g. 'still went through. can "
    f"i get...'); overall error rate {n_errors / len(val):.1%} ({n_errors}/{len(val)})"
)


finalist + char_wb 3-5             acc 0.9031  macroF1 0.8945  weightedF1 0.9029
finalist + balanced                acc 0.9113  macroF1 0.9053  weightedF1 0.9119


In [14]:
# CHECK (5) — leave as is.
if variant_a_name is None or variant_b_name is None or two_intent_note is None:
    print("Task 5 not complete yet: run both variants and write two_intent_note.")
else:
    names = {r["model"] for r in RESULTS}
    assert variant_a_name in names and variant_b_name in names, "record() both variants"
    print(two_intent_note)
    display(results_table().head(8))

of the 69 finalist errors, 63 (91%) look two-intent (proxy: a sentence boundary followed by an uncapitalized letter, since this corpus stitches intents together without re-capitalizing, e.g. 'still went through. can i get...'); overall error rate 9.4% (69/733)


,model,accuracy,macro_f1,weighted_f1,fit_seconds,note
0,finalist + balanced,0.9113,0.9053,0.9119,NaN,
1,mlp,0.9127,0.9051,0.9131,6.8,
2,random forest,0.9113,0.9048,0.9113,1.0,
3,linear svm,0.9113,0.9037,0.9114,0.0,
4,xgboost,0.9045,0.8991,0.9043,7.1,
5,logreg,0.9059,0.8981,0.9059,0.2,
6,tfidf + logreg,0.9059,0.8981,0.9059,NaN,
7,random forest + svd200,0.9004,0.8949,0.9006,1.0,


### Task 5 write-up

Character n grams (analyzer="char_wb", ngram_range=(3,5)) predicted this would help given the typo rate in the corpus, but it made things a bit worse reverted (macro F1 0.8945 vs. the finalist's 0.8981). The likely mechanism, subword fragments overlap across queues (example "ship" appears inside unrelated words too), diluting the sharper word level signals that already separate the queues cleanly, so the extra typo robustness was not worth the added noise. class_weight="balanced" predicted this would help the smallest queue, account access, and it did, everything improved (accuracy 0.9113, macro-F1 0.9053, weighted-F1 0.9119). The mechanism is simple, reweighting the loss to counter class imbalance forces the model to work harder on the minority queue instead of letting it get absorbed into the majority heavy decision boundary. On the two intent cohort, of the finalist's 69 validation errors (a 9.4% overall error rate), 63 look two intent by a sentence boundary proxy (a period or question mark followed by a lowercase letter, since this corpus stitches a second issue on without recapitalizing, instead of using explicit connector words). In production, these are the tickets that should be routed to a human instead of forced into one queue.

---

## Task 6 — Ship it

**6a — the triage policy.** Maya's constraint: auto-route what you can,
send the rest to a human, never be *confidently* wrong. Retrain your winning
configuration on **all of `dedup`** (every labelled ticket you trust), as one
`Pipeline` from raw text so it can be applied to anything. Then, using
**validation predictions from your Task 4/5 model** (not the retrained one —
its training set includes `val` now), build `tradeoff`: for each threshold in
`[0.5, 0.6, 0.7, 0.8, 0.9]`, the fraction of `val` the model would auto-route
(max predicted probability ≥ threshold) and the accuracy **on that
auto-routed fraction**. Choose `AUTO_THRESHOLD` from the table.

**6b — the routing file.** Apply the retrained pipeline to the 400-ticket
`batch`. Build `routed`: `ticket_id`, `predicted_queue`, `confidence` (max
probability, 4 decimals), `needs_human` (True where confidence <
`AUTO_THRESHOLD`). The GIVEN cell validates and writes `routed_batch.csv`.

In [15]:
# (6a): the deployment pipeline and the trade-off table.
final_pipe = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=2000, class_weight="balanced"),
)
final_pipe.fit(dedup["text"], dedup["queue"])

proba_val = model_b.predict_proba(Xva)
confidence_val = proba_val.max(axis=1)
correct_val = (pred_b == y_val.to_numpy())

thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
rows = []
for t in thresholds:
    routed_mask = confidence_val >= t
    rows.append({
        "threshold": t,
        "auto_routed_share": round(routed_mask.mean(), 4),
        "accuracy_on_routed": round(correct_val[routed_mask].mean(), 4),
    })
tradeoff = pd.DataFrame(rows)

AUTO_THRESHOLD = 0.7
threshold_defence = (
    "At 0.7, we auto-route about 75% of tickets and get 96% of those right - "
    "roughly 1 in 25 auto-routed tickets lands in the wrong queue for a day "
    "before anyone notices. Pushing to 0.8 only buys us 0.6 more points of "
    "accuracy but nearly doubles the human queue (47% vs 25% of the inbox), "
    "and a human triaging a flagged ticket costs minutes, not a day, so 0.7 "
    "keeps more of that cost off Maya's team without meaningfully raising risk."
)


In [16]:
# (6b): route the batch.
proba_batch = final_pipe.predict_proba(batch["text"])
predicted_queue = final_pipe.classes_[proba_batch.argmax(axis=1)]
confidence = proba_batch.max(axis=1).round(4)

routed = pd.DataFrame({
    "ticket_id": batch["ticket_id"],
    "predicted_queue": predicted_queue,
    "confidence": confidence,
    "needs_human": confidence < AUTO_THRESHOLD,
})


In [17]:
# GIVEN — do not modify. Validates the routing file and writes it.
if final_pipe is None or tradeoff is None or AUTO_THRESHOLD is None or routed is None:
    print("Task 6 not complete yet: fill in 6a and 6b above.")
else:
    display(tradeoff)
    print(f"chosen AUTO_THRESHOLD = {AUTO_THRESHOLD}\n{threshold_defence}\n")
    assert list(routed.columns) == ["ticket_id", "predicted_queue", "confidence", "needs_human"]
    assert len(routed) == 400 and set(routed["ticket_id"]) == set(batch["ticket_id"])
    assert routed["predicted_queue"].isin(QUEUES).all()
    assert routed["confidence"].between(0, 1).all()
    assert (routed["needs_human"] == (routed["confidence"] < AUTO_THRESHOLD)).all(), \
        "needs_human must be exactly (confidence < AUTO_THRESHOLD)"
    routed.to_csv("routed_batch.csv", index=False)
    n_auto = int((~routed["needs_human"]).sum())
    print(f"routed_batch.csv written: {n_auto} auto-routed ({n_auto / 4:.1f}%), "
          f"{400 - n_auto} flagged for a human.")

,threshold,auto_routed_share,accuracy_on_routed
0,0.5,0.9332,0.9357
1,0.6,0.8718,0.9546
2,0.7,0.7503,0.9600
3,0.8,0.5280,0.9664
4,0.9,0.2115,0.9871


chosen AUTO_THRESHOLD = 0.7
At 0.7, we auto-route about 75% of tickets and get 96% of those right - roughly 1 in 25 auto-routed tickets lands in the wrong queue for a day before anyone notices. Pushing to 0.8 only buys us 0.6 more points of accuracy but nearly doubles the human queue (47% vs 25% of the inbox), and a human triaging a flagged ticket costs minutes, not a day, so 0.7 keeps more of that cost off Maya's team without meaningfully raising risk.

routed_batch.csv written: 328 auto-routed (82.0%), 72 flagged for a human.


## Task 7 — The memo

Subject: Ticket router, ready to ship, here's what to expect

Author: Hesham Rashid

Maya, here's where the router stands.

What it does: It reads each incoming ticket's text and automatically assigns it to one of your five queues (shipping, returns/warranty, billing, account access, product question), and for any ticket it isn't confident about, it hands that one to a person instead of guessing.

Accuracy on what it auto routes: On held out tickets, at the confidence bar we've set, the router gets about 96% of its auto-routed tickets right, about 1 in 25 lands in the wrong queue and could sit there for a day before someone notices.

The triage split: On this morning's 400 ticket batch, it auto routed 328 tickets (82%) and flagged 72 (18%) for a human. That's by design, it would rather over flag borderline cases than confidently misroute them.

What it can't do: It only ever picks one queue per ticket, so when a customer raises two problems in the same message (a return and a billing question, say), the router usually catches the more prominent one and can miss the other, that pattern shows up in the large majority of its mistakes.

A risk to watch: The model was trained on a snapshot of today's ticket mix, so if that mix shifts, like a new product line, a holiday return surge, a new complaint pattern, etc. it's accuracy could quietly drift without anyone noticing. I would recommend tracking the weekly auto routed share and average confidence per queue, and spot-checking a small sample of auto routed tickets monthly against their true queue.

The number that gives me pause: Overall model accuracy across all tickets is about 91%, and it does worse on account access, our smallest queue, so this is a first router, not a finished one, and it's worth revisiting once we've watched it run for a few weeks.

---

In [18]:
# GIVEN — do not modify. Final tally: what a marker will find.
import os

done = {
    "Task 1 tables": task1_table is not None,
    "Task 2 honest split": val is not None and n_straddle is not None,
    "Task 3 baselines + misroutes": misroutes is not None,
    "Task 4 bake-off (>= 10 registry rows)": len(RESULTS) >= 10,
    "Task 5 two variants + cohort note": two_intent_note is not None,
    "Task 6 routed_batch.csv on disk": os.path.exists("routed_batch.csv"),
}
for k, v in done.items():
    print(f"  {'[x]' if v else '[ ]'} {k}")
if all(done.values()):
    print("\nAll structural checks pass. Before submitting: Restart & Run All, "
          "confirm every write-up cell no longer contains its italic prompt, "
          "and attach routed_batch.csv and your AI Usage Report in Canvas.")
else:
    print("\nStructural gaps remain - see the unchecked boxes above.")

  [x] Task 1 tables
  [x] Task 2 honest split
  [x] Task 3 baselines + misroutes
  [x] Task 4 bake-off (>= 10 registry rows)
  [x] Task 5 two variants + cohort note
  [x] Task 6 routed_batch.csv on disk

All structural checks pass. Before submitting: Restart & Run All, confirm every write-up cell no longer contains its italic prompt, and attach routed_batch.csv and your AI Usage Report in Canvas.
